In [1]:
import pandas as pd
import numpy as np
from pathlib import Path
from math import sqrt, erf

ROOT = Path().resolve().parent
DATA_PATH = ROOT / "data_raw"

print("Directorio actual:", ROOT)
print("Ruta de datos:", DATA_PATH)

Directorio actual: /Users/gabrielbohorquez/Desktop/Ironhack/Semana_5/PROYECTO_2/Projecto-2-Vanguard-ab-test
Ruta de datos: /Users/gabrielbohorquez/Desktop/Ironhack/Semana_5/PROYECTO_2/Projecto-2-Vanguard-ab-test/data_raw


In [2]:
# Cargar datos limpios
df_demo       = pd.read_csv(DATA_PATH / "df_demo_clean.csv")
df_experiment = pd.read_csv(DATA_PATH / "df_experiment_clean.csv")
df_web        = pd.read_csv(DATA_PATH / "df_web_clean.csv")

print("Datasets cargados correctamente!")
print("df_demo_clean:      ", len(df_demo), "filas")
print("df_experiment_clean:", len(df_experiment), "filas")
print("df_web_clean:       ", len(df_web), "filas")

Datasets cargados correctamente!
df_demo_clean:       70591 filas
df_experiment_clean: 50500 filas
df_web_clean:        744641 filas


In [3]:
# Cargar datos limpios
import pandas as pd

def inspect_datasets(dataframes: list[pd.DataFrame], names: list[str]) -> None:
    """
    Imprime las columnas de cada dataset para verificar su estructura.
    """
    for df, name in zip(dataframes, names):
        print(f"\n--- Columnas en {name} ---")
        print(list(df.columns))

# Lista de tus DataFrames y sus nombres para el bucle
datasets = [df_demo, df_experiment, df_web]
dataset_names = ["df_demo", "df_experiment", "df_web"]

# Ejecutar la inspección
inspect_datasets(datasets, dataset_names)




--- Columnas en df_demo ---
['client_id', 'clnt_tenure_yr', 'clnt_tenure_mnth', 'clnt_age', 'gendr', 'num_accts', 'bal', 'calls_6_mnth', 'logons_6_mnth']

--- Columnas en df_experiment ---
['client_id', 'Variation']

--- Columnas en df_web ---
['client_id', 'visitor_id', 'visit_id', 'process_step', 'date_time']


In [4]:
def check_content(dataframes: list[pd.DataFrame], names: list[str]) -> None:
    """
    Muestra las primeras filas de cada dataset de forma organizada.
    """
    for df, name in zip(dataframes, names):
        print(f"\n{'='*10} Contenido de {name} {'='*10}")
        # Usamos display(df.head()) si estás en Jupyter, o print() en script puro
        print(df.head(3)) 

# Ejecutar la revisión
datasets = [df_demo, df_experiment, df_web]
dataset_names = ["df_demo", "df_experiment", "df_web"]

check_content(datasets, dataset_names)


========== Contenido de df_demo ==========
   client_id  clnt_tenure_yr  clnt_tenure_mnth  clnt_age gendr  num_accts  \
0     836976             6.0              73.0      60.5     U        2.0   
1    2304905             7.0              94.0      58.0     U        2.0   
2    1439522             5.0              64.0      32.0     U        2.0   

         bal  calls_6_mnth  logons_6_mnth  
0   45105.30           6.0            9.0  
1  110860.30           6.0            9.0  
2   52467.79           6.0            9.0  

========== Contenido de df_experiment ==========
   client_id Variation
0    9988021      Test
1    8320017      Test
2    4033851   Control

========== Contenido de df_web ==========
   client_id            visitor_id                      visit_id process_step  \
0    9988021  580560515_7732621733  781255054_21935453173_531117       step_3   
1    9988021  580560515_7732621733  781255054_21935453173_531117       step_2   
2    9988021  580560515_7732621733  7812550

# 1 analysis de la efectividad del rediseño.

Usaremos df_experiment para separar a los usuarios en Test (rediseño) y Control (diseño viejo) y lo cruzaremss con df_web para ver quién completó el proceso.

A. Completion Rate (Tasa de Finalización)

Hipótesis Nula (H_0): No hay diferencia en la tasa de finalización entre el grupo de Test y el de Control.

Hipótesis Alternativa (H_1): El nuevo diseño (Test) tiene una tasa de finalización significativamente mayor.

Método: Realizaremos un test para proporciones. Si el p-value es menor a 0.05, rechazamos H_0 y confirmamos que el rediseño es efectivo.

In [5]:
# hipotisis testing: Tasa de Finalización
# Identificamos quién llegó al paso 'confirm'
df_web['is_completed'] = (df_web['process_step'] == 'confirm').astype(int)

# Agrupamos por cliente (1 si completó al menos una vez, 0 si nunca)
df_conversion = df_web.groupby('client_id')['is_completed'].max().reset_index()

# Unimos con el experimento
df_test_logic = pd.merge(df_conversion, df_experiment, on='client_id')

# Separamos los grupos
group_control = df_test_logic[df_test_logic['Variation'] == 'Control']['is_completed']
group_test = df_test_logic[df_test_logic['Variation'] == 'Test']['is_completed']

In [6]:
# Z-test manual para diferencia de proporciones: Completion Rate Test vs Control

control_success = group_control.sum()
control_n = len(group_control)

test_success = group_test.sum()
test_n = len(group_test)

control_rate = control_success / control_n
test_rate = test_success / test_n

absolute_diff = test_rate - control_rate
relative_lift = absolute_diff / control_rate

pooled_rate = (control_success + test_success) / (control_n + test_n)
standard_error = sqrt(
    pooled_rate * (1 - pooled_rate) * ((1 / control_n) + (1 / test_n))
)

z_stat = absolute_diff / standard_error

# P-value unilateral: H1 = Test > Control
p_value_one_sided = 1 - (0.5 * (1 + erf(z_stat / sqrt(2))))

# P-value bilateral
p_value_two_sided = 2 * min(p_value_one_sided, 1 - p_value_one_sided)

print("=== COMPLETION RATE Z-TEST ===")
print(f"Control successes: {control_success}")
print(f"Control total: {control_n}")
print(f"Control rate: {control_rate:.2%}")
print(f"Test successes: {test_success}")
print(f"Test total: {test_n}")
print(f"Test rate: {test_rate:.2%}")
print(f"Diferencia absoluta Test - Control: {absolute_diff:.2%}")
print(f"Incremento relativo: {relative_lift:.2%}")
print(f"Z-statistic: {z_stat:.4f}")
print(f"P-value unilateral: {p_value_one_sided:.10f}")
print(f"P-value bilateral: {p_value_two_sided:.10f}")

if p_value_one_sided < 0.05:
    print("Decisión: se rechaza H0. El Completion Rate es significativamente mayor en Test.")
else:
    print("Decisión: no se rechaza H0. No hay evidencia suficiente de mejora en Test.")

=== COMPLETION RATE Z-TEST ===
Control successes: 15434
Control total: 23532
Control rate: 65.59%
Test successes: 18687
Test total: 26968
Test rate: 69.29%
Diferencia absoluta Test - Control: 3.71%
Incremento relativo: 5.65%
Z-statistic: 8.8745
P-value unilateral: 0.0000000000
P-value bilateral: 0.0000000000
Decisión: se rechaza H0. El Completion Rate es significativamente mayor en Test.


## Conclusión del test de hipótesis: Completion Rate

El test Z de diferencia de proporciones muestra que el grupo Test tiene una tasa de finalización significativamente mayor que el grupo Control.

Resultados principales:

- Control Completion Rate: 65,59 %
- Test Completion Rate: 69,29 %
- Diferencia absoluta: +3,71 puntos porcentuales
- Incremento relativo: +5,65 %
- Z-statistic: 8,8745
- P-value unilateral: < 0,05
- P-value bilateral: < 0,05

Dado que el p-value es menor que 0,05, se rechaza la hipótesis nula. Existe evidencia estadística suficiente para concluir que el rediseño Test mejora la tasa de finalización del proceso frente al diseño Control.

Desde una perspectiva de negocio, el rediseño muestra una mejora real en conversión. Sin embargo, esta conclusión debe interpretarse junto con las métricas de fricción del usuario, especialmente el Client Backtracking Rate.


B. Cost-Effectiveness Threshold

Aquí no solo buscamos que sea "mejor", sino que supere un umbral (ej. un aumento del 5%.
Análisis: Si el coste de implementar el rediseño es alto, la mejora observada debe justificar la inversión.

In [ ]:
# hypothesis testing: Cost-Effectiveness Threshold
# Definimos el umbral (5%)

threshold = 0.05

# Calculamos las tasas de finalización (medias)
rate_control = group_control.mean()
rate_test = group_test.mean()

# Calculamos la mejora absoluta
observed_diff = rate_test - rate_control

print(f"Mejora observada: {observed_diff:.2%}")

Mejora observada: 3.71%


In [8]:
observed_diff = rate_test - rate_control
print(f"Mejora observada: {observed_diff:.2%}")
print(f"Umbral requerido: {threshold:.2%}")

if observed_diff >= threshold:
    print("El rediseño supera el umbral de rentabilidad.")
else:
    print(f"El rediseño NO supera el umbral: {observed_diff:.2%} < {threshold:.2%}")

Mejora observada: 3.71%
Umbral requerido: 5.00%
El rediseño NO supera el umbral: 3.71% < 5.00%


## Conclusión del umbral de negocio

Aunque el grupo Test muestra una mejora estadísticamente significativa en Completion Rate, la mejora observada es de +3,71 puntos porcentuales.

El umbral mínimo definido para considerar el rediseño suficientemente rentable era de +5,00 puntos porcentuales.

Por tanto, desde una perspectiva de negocio, el rediseño mejora la conversión, pero no alcanza por sí solo el umbral mínimo definido para justificar una implementación definitiva si el coste de despliegue es elevado.

Esta conclusión refuerza la necesidad de interpretar el resultado junto con las métricas de fricción del usuario antes de tomar una decisión final.

C. Edad y Tenencia 

Dado que tenemos df_demo, podemos verificar si el rediseño funciona mejor para ciertos segmentos:

¿Influye la edad (clnt_age) en la probabilidad de completar el proceso?

¿Los clientes más antiguos (clnt_tenure_yr) son más resistentes al cambio que los nuevos?

In [9]:
# hypothesis testing: Edad y Tenencia 
# Unimos los datos de conversión (que ya calculamos) con df_demo y df_experiment
df_segmented = pd.merge(df_conversion, df_demo, on='client_id')
df_segmented = pd.merge(df_segmented, df_experiment, on='client_id')

# Filtramos por los grupos de interés
test_group = df_segmented[df_segmented['Variation'] == 'Test']
control_group = df_segmented[df_segmented['Variation'] == 'Control']

In [11]:
# Comparación descriptiva de edad: completaron vs no completaron en el grupo Test

completed_age = test_group[test_group["is_completed"] == 1]["clnt_age"]
not_completed_age = test_group[test_group["is_completed"] == 0]["clnt_age"]

age_summary = pd.DataFrame({
    "group": ["Completed", "Not completed"],
    "mean_age": [completed_age.mean(), not_completed_age.mean()],
    "median_age": [completed_age.median(), not_completed_age.median()],
    "users": [completed_age.count(), not_completed_age.count()]
})

age_difference = completed_age.mean() - not_completed_age.mean()

print("=== AGE COMPARISON - TEST GROUP ===")
print(age_summary)
print(f"Diferencia de edad media Completed - Not completed: {age_difference:.2f} años")

=== AGE COMPARISON - TEST GROUP ===
           group   mean_age  median_age  users
0      Completed  46.567418        47.0  18682
1  Not completed  48.513713        50.0   8277
Diferencia de edad media Completed - Not completed: -1.95 años


In [12]:
# Comparación descriptiva de antigüedad: usuarios que no completaron en Test vs Control

tenure_fail_test = test_group[test_group["is_completed"] == 0]["clnt_tenure_yr"]
tenure_fail_control = control_group[control_group["is_completed"] == 0]["clnt_tenure_yr"]

tenure_summary = pd.DataFrame({
    "group": ["Test not completed", "Control not completed"],
    "mean_tenure_yr": [tenure_fail_test.mean(), tenure_fail_control.mean()],
    "median_tenure_yr": [tenure_fail_test.median(), tenure_fail_control.median()],
    "users": [tenure_fail_test.count(), tenure_fail_control.count()]
})

tenure_difference = tenure_fail_test.mean() - tenure_fail_control.mean()

print("=== TENURE COMPARISON - NOT COMPLETED USERS ===")
print(tenure_summary)
print(f"Diferencia de antigüedad media Test - Control: {tenure_difference:.2f} años")

=== TENURE COMPARISON - NOT COMPLETED USERS ===
                   group  mean_tenure_yr  median_tenure_yr  users
0     Test not completed       12.066087              11.0   8277
1  Control not completed       12.170536              11.0   8098
Diferencia de antigüedad media Test - Control: -0.10 años


## Conclusión de segmentación por edad y antigüedad

En el grupo Test, los usuarios que completaron el proceso tienen una edad media de 46,57 años, mientras que los usuarios que no completaron tienen una edad media de 48,52 años.

Esto sugiere que los usuarios que no completan tienden a ser ligeramente mayores, con una diferencia media de 1,95 años. La diferencia no debe interpretarse como causal, pero sí como una señal útil para revisar si la nueva interfaz genera más fricción en determinados segmentos de edad.

En cuanto a la antigüedad, los usuarios que no completaron en Test tienen una antigüedad media de 12,07 años, frente a 12,17 años en Control. La diferencia es de apenas -0,10 años, por lo que no parece haber una diferencia relevante de antigüedad entre los usuarios que abandonan en ambos grupos.

Conclusión: la edad parece aportar una señal descriptiva más útil que la antigüedad. Para una recomendación de negocio, conviene revisar especialmente la experiencia de usuarios de mayor edad dentro del rediseño Test.